# 03.6 — The pipeline, and what it bought

Five notebooks of pieces. This one assembles them into something that runs
unattended, then measures whether any of it helped.

The predictions from notebook 5, written down before we run anything:

1. Twelve questions become reachable — the ceiling moves from 0.667 to 1.0
2. The seven staleness failures should convert
3. **Q12 and Q13 must not break.** They depend on the amended circular, and if
   the amendment logic is wrong they'll disappear inside a net improvement.

In [1]:
# Install if using jupyterlab locally on cpu. Install before install sentence-transformers
!pip install -q torch==2.14.0+cpu --index-url https://download.pytorch.org/whl/cpu

In [2]:
!pip install -q pymupdf4llm==1.28.2 python-docx==1.2.0 openpyxl==3.1.5 python-pptx==1.0.2 \
               beautifulsoup4==4.15.0 lxml==6.1.3 ftfy==6.3.1 pytesseract==0.3.13 pdf2image==1.17.0 sentence-transformers==6.0.1

In [3]:
import shutil, subprocess, os

if not shutil.which('tesseract') or not shutil.which('pdftoppm'):
    sudo = [] if os.geteuid() == 0 else ['sudo']
    subprocess.run([*sudo, 'apt-get', 'update', '-qq'])
    subprocess.run([*sudo, 'apt-get', 'install', '-y', '-qq',
                    'tesseract-ocr', 'poppler-utils'])

print('tesseract:', shutil.which('tesseract'))
print('poppler  :', shutil.which('pdftoppm'))

debconf: delaying package configuration, since apt-utils is not installed


Selecting previously unselected package poppler-data.
(Reading database ... 5835 files and directories currently installed.)
Preparing to unpack .../00-poppler-data_0.4.12-1_all.deb ...
Unpacking poppler-data (0.4.12-1) ...
Selecting previously unselected package krb5-locales.
Preparing to unpack .../01-krb5-locales_1.20.1-6ubuntu2.8_all.deb ...
Unpacking krb5-locales (1.20.1-6ubuntu2.8) ...
Selecting previously unselected package libbsd0:amd64.
Preparing to unpack .../02-libbsd0_0.12.1-1build1.1_amd64.deb ...
Unpacking libbsd0:amd64 (0.12.1-1build1.1) ...
Selecting previously unselected package libexpat1:amd64.
Preparing to unpack .../03-libexpat1_2.6.1-2ubuntu0.4_amd64.deb ...
Unpacking libexpat1:amd64 (2.6.1-2ubuntu0.4) ...
Selecting previously unselected package libfribidi0:amd64.
Preparing to unpack .../04-libfribidi0_1.0.13-3build1_amd64.deb ...
Unpacking libfribidi0:amd64 (1.0.13-3build1) ...
Selecting previously unselected package libglib2.0-0t64:amd64.
Preparing to unpack .../

## The handlers

Notebooks 1 and 3, condensed. The only new thing is that `read_pdf` now checks
for a text layer and routes to OCR when there isn't one.

In [4]:
import re, csv, zipfile
from pathlib import Path
from email import policy
from email.parser import BytesParser

import pymupdf, pymupdf4llm
from bs4 import BeautifulSoup
from lxml import etree
from openpyxl import load_workbook
from pptx import Presentation

CORPUS = Path('../../corpus/docs')
W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'


def read_pdf(path):
    with pymupdf.open(path) as doc:
        fonts = sum(len(page.get_fonts()) for page in doc)
    if fonts == 0:
        import pytesseract
        from pdf2image import convert_from_path
        pages = convert_from_path(path, dpi=200)
        return '\n'.join(pytesseract.image_to_string(p) for p in pages), 'ocr'
    return pymupdf4llm.to_markdown(str(path)), 'text'

def read_docx(path):
    root = etree.fromstring(zipfile.ZipFile(path).read('word/document.xml'))
    for deleted in root.iter(f'{W}del'):
        deleted.getparent().remove(deleted)
    paras = (''.join(t.text or '' for t in p.iter(f'{W}t')) for p in root.iter(f'{W}p'))
    return '\n'.join(p for p in paras if p.strip()), 'text'

def read_xlsx(path):
    out = []
    for sheet in load_workbook(path, data_only=True).worksheets:
        if sheet.sheet_state != 'visible':
            continue
        out.append(f'## {sheet.title}')
        for row in sheet.iter_rows():
            cells = [str(c.value) if c.value is not None else '' for c in row]
            if any(cells):
                out.append(' | '.join(cells).rstrip(' |'))
    return '\n'.join(out), 'text'

def read_pptx(path):
    parts = []
    for i, slide in enumerate(Presentation(path).slides, 1):
        body = [sh.text_frame.text for sh in slide.shapes
               if sh.has_text_frame and sh.text_frame.text.strip()]
        parts.append(f'## Slide {i}\n' + '\n'.join(body))
        if slide.has_notes_slide and slide.notes_slide.notes_text_frame.text.strip():
            parts.append('[speaker notes]\n' + slide.notes_slide.notes_text_frame.text.strip())
    return '\n\n'.join(parts), 'text'

def read_html(path):
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'lxml')
    for tag in soup(['nav', 'header', 'footer', 'aside', 'script', 'style', 'form']):
        tag.decompose()
    return (soup.find('main') or soup.body).get_text('\n', strip=True), 'text'

def read_eml(path):
    msg = BytesParser(policy=policy.default).parse(path.open('rb'))
    body = msg.get_body(preferencelist=('plain',)).get_content()
    kept = []
    for line in body.splitlines():
        if line.lstrip().startswith('>'):
            break
        kept.append(line)
    header = f"From: {msg['From']}\nDate: {msg['Date']}\nSubject: {msg['Subject']}"
    return header + '\n\n' + '\n'.join(kept).strip(), 'text'


def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        rows = [r for r in csv.reader(f) if r and not r[0].lstrip().startswith('#')]
    header, *body = rows
    return '\n\n'.join('\n'.join(f'{h}: {v}' for h, v in zip(header, r))
                       for r in body), 'text'

HANDLERS = {'.pdf': read_pdf, '.docx': read_docx, '.xlsx': read_xlsx,
            '.pptx': read_pptx, '.html': read_html, '.eml': read_eml,
            '.csv': read_csv,
            '.md': lambda p: (p.read_text(encoding='utf-8'), 'text')}

print(f'{len(HANDLERS)} handlers')

8 handlers


## Clean, structure, metadata

Notebooks 3, 4 and 5, condensed.

Note what the metadata step is and isn't. There's no extraction here — no regex
hunting for reference numbers or effective dates in the text. The register is a
CSV that somebody maintains, and the pipeline reads it.

That file is two rows for fifteen documents, because currency is a property of
document *families*, not of files.

In [5]:
import ftfy

BOILERPLATE = re.compile(
    r'SYNTHETIC DOCUMENT.*?(?:cite as fact\.|Not a real record\.)', re.DOTALL)


def clean(text):
    text = ftfy.fix_text(text)
    text = BOILERPLATE.sub('', text).replace('~~', '')
    text = re.sub(r'  +', ' ', text)
    out, blank = [], False
    for line in text.splitlines():
        if line.strip():
            out.append(line.rstrip()); blank = False
        elif not blank:
            out.append(''); blank = True
    return '\n'.join(out).strip()

def sections(markdown):
    stack, out, buf = {}, [], []

    def flush():
        body = '\n'.join(buf).strip()
        if body:
            out.append((' > '.join(stack[k] for k in sorted(stack)), body))
        buf.clear()

    for line in markdown.splitlines():
        heading = re.match(r'^(#{1,6})\s+(.*)$', line.strip())
        if heading:
            flush()
            level = len(heading.group(1))
            stack[level] = re.sub(r'[*_`]|<[^>]+>', '', heading.group(2)).strip()
            for deeper in [k for k in stack if k > level]:
                del stack[deeper]
        else:
            buf.append(line)
    flush()
    return out or [('', markdown)]

# Notebook 5: the register is DATA, maintained by whoever owns the documents.
# Nothing here parses the text looking for version numbers.
import csv

REGISTER = {
    row['document']: (row['status'], row['superseded_by'])
    for row in csv.DictReader(
        open('../../corpus/document-register.csv', newline='', encoding='utf-8'))
}

for doc, (status, by) in REGISTER.items():
    print(f'{status:<12} {doc}')

superseded   sahel-employee-handbook-2023.pdf
amended      nfsc-circular-2024-07-cybersecurity.pdf


## One document, end to end

`ingest()` is the whole module in twenty lines.

In [6]:
import hashlib
from datetime import datetime, timezone

def chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]

def base_metadata(path, method):
    """The four fields any corpus needs, plus a content hash."""
    return {
        'source': path.name,
        'content_hash': hashlib.sha256(path.read_bytes()).hexdigest()[:16],
        'ingested_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'extraction': method,
        'extracted_by': 'pymupdf4llm==1.28.2',
    }

def corpus_metadata(name):
    """Specific to this corpus: it has a currency problem, so it has a status.
    A corpus of research papers would return year, venue and doi instead."""
    status, superseded_by = REGISTER.get(name, ('current', None))
    return {'status': status, 'superseded_by': superseded_by}


def ingest(path):
    handler = HANDLERS.get(path.suffix.lower())
    if handler is None:
        raise ValueError(f'no handler for {path.suffix}')

    raw, method = handler(path)
    if not raw.strip():
        raise ValueError('parser returned nothing')    # fail loudly, not silently

    text = clean(raw)
    meta = {**base_metadata(path, method), **corpus_metadata(path.name)}

    chunks = []
    for heading, body in sections(text):
        for i, piece in enumerate(chunk(body)):
            if not piece.strip():
                continue
            prefix = f'[{path.name} | {heading}]\n' if heading else ''
            chunks.append({**meta, 'heading': heading, 'n': i, 'text': prefix + piece})
    return chunks

for c in ingest(CORPUS / 'sahel-employee-handbook-2025.pdf'):
    if '25 working days' in c['text']:
        print(c['text'][:180])
        print()
        for k, v in c.items():
            if k != 'text':
                print(f'   {k:<15} {v}')


=== Document parser messages ===
Using Tesseract for OCR processing.
[sahel-employee-handbook-2025.pdf | Sahel Microfinance Bank Plc > 4. Annual leave]
Confirmed staff are entitled to **25 working days** of paid annual leave each calendar year, excl

   source          sahel-employee-handbook-2025.pdf
   content_hash    81512862a2d90715
   ingested_at     2026-09-12T18:08:40+00:00
   extraction      text
   extracted_by    pymupdf4llm==1.28.2
   status          current
   superseded_by   None
   heading         Sahel Microfinance Bank Plc > 4. Annual leave
   n               0


The chunk carries its text, its heading path, and the metadata record from
notebook 5 — five universal fields and two that exist because this corpus has a
currency problem.

Swap the corpus and `corpus_metadata` is the only function you rewrite.

In [7]:
import json, hashlib

STATE = Path('ingest-state.json')


def fingerprint(path):
    """Content hash — so an unchanged file is skipped even if it was touched,
    and a changed one is re-ingested even if its name and date are identical."""
    return hashlib.sha256(path.read_bytes()).hexdigest()[:16]


def run(corpus, state_file=STATE, force=False):
    done = {} if force or not state_file.exists() else json.loads(state_file.read_text())
    chunks, failures, skipped = [], [], 0

    for path in sorted(corpus.iterdir()):
        if not path.is_file:
            continue
        fp = fingerprint(path)
        if done.get(path.name) == fp:
            skipped += 1
            continue
        try:
            produced = ingest(path)
            chunks.extend(produced)
            done[path.name] = fp
            print(f'  {len(produced):>3} chunks  {path.name}')
        except Exception as exc:
            failures.append({'file': path.name, 'error': f'{type(exc).__name__}: {exc}'})
            print(f'  FAILED  {path.name}  {type(exc).__name__}')

    state_file.write_text(json.dumps(done, indent=2))
    return chunks, failures, skipped

chunks, failures, skipped = run(CORPUS, force=True)

print(f'\n{len(chunks)} chunks | {skipped} skipped | {len(failures)} failed')
for f in failures:
    print(' ', f)


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
   14 chunks  kaduna-agro-annual-report-2024.pdf
    5 chunks  kaduna-agro-board-deck-2025-01.pptx

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
   14 chunks  kaduna-agro-board-minutes-2024-10-17.pdf
    3 chunks  kaduna-agro-distribution-2024.xlsx
    4 chunks  kdirs-guidance-note-4-2024-SCANNED.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
   13 chunks  nfsc-circular-2024-07-cybersecurity.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
    8 chunks  nfsc-circular-2025-02-amendment.pdf
    3 chunks  nfsc-circular-2025-02.html
    2 chunks  sahel-approved-vendors.csv
    5 chunks  sahel-branch-runbook.md

=== Document parser messages ===
Using Tesseract for OCR processing.
   19 chunks  sahel-employee-handbook-2023.pdf

=== Document parser messages ===
Using Tesseract for OCR processing.
   19

Run that cell again and every document is skipped — the fingerprints match.

The fingerprint is a content hash rather than a modification time, deliberately.
Copying files between systems rewrites timestamps, so a mtime check re-ingests
documents that haven't changed and misses ones that have.

Notice also what `failures` is: a list you can inspect, not a crash. In a real
pipeline that list goes somewhere a human sees it. **A document that failed to
ingest and nobody noticed is the same as a document that doesn't exist** — and
the only sign is a question that quietly stops working.

In [8]:
chunks_again, failures_again, skipped_again = run(CORPUS)
print(f'second run: {len(chunks_again)} new chunks, {skipped_again} skipped')

second run: 0 new chunks, 15 skipped


## Where documents come from

Everything so far assumed a folder. Real engagements don't start there.

The documents are in Google Drive, or SharePoint, or Notion, or a shared inbox,
or three of those at once. Getting them out is a connector, and the libraries
exist — Unstructured ships them, LangChain and LlamaIndex have loaders, Airbyte
does it at scale.

Three things matter more than which one you pick.

**Incremental sync.** Re-downloading a document store nightly stops being viable
quickly. You want changes since the last run, which means the source has to
expose modification times you can trust, and often it doesn't.

**Permissions travel with the document.** If a file is restricted to the finance
team, that restriction has to reach your metadata, or your RAG system becomes a
way to read documents you can't open. This is the most common serious security
failure in enterprise RAG and it's an ingestion bug.

**Connectors skip things silently.** A connector that can't read a file type, or
hits a permissions error on one folder, frequently logs nothing and returns fewer
documents. Same failure class as the silent truncation and the empty parser
result — and the defence is the same: count what you expected and compare.

## Now measure it

Everything above is engineering. This is the part that tells you whether it
worked.

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Drop superseded documents. Keep 'amended' ones — still in force for
# everything the amendment didn't change. Collapsing these two statuses is
# what would break Q12 and Q13.
live = [c for c in chunks if c['status'] != 'superseded']

from collections import Counter
print('chunks by status:', dict(Counter(c['status'] for c in chunks)))
print(f'indexing {len(live)} of {len(chunks)}')

embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')
vectors = embedder.encode([c['text'] for c in live], normalize_embeddings=True)

def search(query, k=5):
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    return [live[i] for i in np.argsort(-(vectors @ qv))[:k]]

chunks by status: {'current': 93, 'amended': 13, 'superseded': 19}
indexing 106 of 125


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
questions = list(csv.DictReader(
    open('../../corpus/golden_questions.csv', newline='', encoding='utf-8')))
scoreable = [q for q in questions if q['source_document'].strip()]

K = 1
results = []
for q in scoreable:
    returned = [c['source'] for c in search(q['question'], 5)]
    results.append({'id': q['id'], 'question': q['question'],
                   'expected': q['source_document'], 'returned': returned,
                   'difficulty': q['difficulty'], 'failure_class': q['failure_class'],
                   'hit': q['source_document'] in returned[:K]})

score = sum(r['hit'] for r in results) / len(results)
print(f'hit@{K}: {score:.3f} ({sum(r["hit"] for r in results)}/{len(results)})')

hit@1: 0.750 (27/36)


## Against the predictions

In [11]:
baseline = json.loads(Path('../../results/02-baseline.json').read_text())
before = {r['id']: r['hit'] for r in baseline['results']}
after = {r['id']: r['hit'] for r in results}

print(f"module 02: {baseline['score']:.3f}")
print(f"module 03: {score:.3f}")
print()

gained = sorted(i for i in after if after[i] and not before.get(i))
lost = sorted(i for i in after if not after[i] and before.get(i))

print(f'now passing ({len(gained)}): {gained}')
print(f'now FAILING ({len(lost)}): {lost}')

print('\nthe two that must not break:')
for qid in ('Q12', 'Q13'):
    print(f'  {qid}: {before.get(qid)} -> {after.get(qid)}')

module 02: 0.444
module 03: 0.750

now passing (14): ['Q02', 'Q03', 'Q04', 'Q06', 'Q07', 'Q11', 'Q14', 'Q24', 'Q28', 'Q31', 'Q32', 'Q34', 'Q37', 'Q38']
now FAILING (3): ['Q05', 'Q09', 'Q10']

the two that must not break:
  Q12: True -> True
  Q13: True -> True


In [12]:
print('by failure class:\n')
classes, hits = Counter(), Counter()
for r in results:
    for cls in r['failure_class'].split('+'):
        classes[cls] += 1
        hits[cls] += r['hit']
for cls in sorted(classes):
    print(f'  {cls:<24} {hits[cls]}/{classes[cls]}')

print('\nstill failing:')
for r in results:
    if not r['hit']:
        print(f"  {r['id']}  [{r['failure_class']}]")
        print(f"      wanted {r['expected']}")
        print(f"      got    {r['returned'][0]}")

by failure class:

  amendment                0/3
  attribution              2/2
  chart-only               2/2
  clarification            1/1
  email-thread             1/2
  embedded-newlines        0/1
  formula-no-cache         0/1
  hidden-sheet             0/1
  html-boilerplate         1/1
  lookup                   4/4
  multi-fact               0/1
  negation                 5/5
  ocr-required             1/1
  page-spanning-table      1/1
  speaker-notes-only       2/2
  staleness                5/6
  table                    0/1
  table-band               2/2
  tracked-changes          1/1
  trailing-note            0/1
  trap                     1/1

still failing:
  Q05  [staleness+multi-fact]
      wanted sahel-employee-handbook-2025.pdf
      got    sahel-hr-memo-2024-41-TRACKED.docx
  Q08  [amendment]
      wanted nfsc-circular-2025-02-amendment.pdf
      got    nfsc-circular-2024-07-cybersecurity.pdf
  Q09  [amendment]
      wanted nfsc-circular-2025-02-amendment.pdf
 

In [13]:
out = Path('../../results/03-ingestion.json')
out.write_text(json.dumps({
    'label': 'module-03 ingestion',
    'metric': f'hit@{K}',
    'score': round(score, 4),
    'n': len(results),
    'pipeline': {'documents': 15, 'chunks_total': len(chunks),
                'chunks_indexed': len(live), 'chunk_size': 500,
                'embedding_model': 'BAAI/bge-small-en-v1.5',
                'heading_context': True,
                'register_entries': len(REGISTER)},
    'results': results,
}, indent=2))
print('saved', out)

saved ../../results/03-ingestion.json


## Reading the result honestly

Whatever number you got, three things are worth checking before you believe it.

**Did anything regress?** The `now FAILING` list is the one that matters. A net
gain can hide two working questions breaking, and notebook 5 showed exactly how.

**Is the gain where you predicted?** Twelve questions were unreachable and should
now be reachable. If the score rose by less than that, something in the new
documents isn't retrieving well, and the failure-class breakdown says which.

**How much of this is real?** Thirty-six questions. A three-point move is roughly
one question. You changed six things at once in this module — new formats, OCR,
cleaning, heading context, metadata, supersession — and this scorer cannot tell
you which of them earned the improvement.

That's not a flaw you can fix by squinting harder at the number. It needs a
harness that can isolate one change at a time, and confidence intervals to say
whether a difference is real at all.

Module 06.

## End of module 03

Fifteen documents, eight formats, one of them a photocopy. Text that's been
repaired, de-boilerplated and kept in its structural context. Every chunk knowing
where it came from and whether that source is still in force.

The thing worth carrying forward isn't any of the individual techniques. It's
that every failure in this module was silent. The parser that returned nothing,
the library that dropped a tracked change, the OCR setting that lost one phrase,
the boilerplate stripper that would have deleted a policy sentence, the
supersession rule that would have broken two working questions.

None of them raised an error. All of them were found by looking at the output or
by measuring against known answers.

**Next:** module 04, chunking — where the 500 in `chunk(text, size=500)` finally
gets justified, or doesn't.